# Testing the code with sample data

In [4]:
import json
import pandas as pd

# Loading the sample data
with open('../data/raw/sample_candidates.json', 'r', encoding='utf-8') as f:
    sample_data = json.load(f)

# 2. candidate's key
first_candidate = sample_data[0]
profile = first_candidate.get('profile', {})
print("=== Candidate Root Keys ===")
print(first_candidate.keys())

# 3. Summary
print("=== Candidate Quick Profile ===")
print(f"ID: {first_candidate.get('candidate_id')}")
print(f"Experience Years: {profile.get('years_of_experience')}")
print(f"Current Location: {profile.get('location')}")
print(f"Willing to Relocate: {first_candidate.get('willing_to_relocate')}")

# 4. Checking what the redrob_signals object looks like inside
print("\n=== redrob_signals ===")
signals = first_candidate.get('redrob_signals', {})
# first 5 signals to see
print(dict(list(signals.items())[:5]))

=== Candidate Root Keys ===
dict_keys(['candidate_id', 'profile', 'career_history', 'education', 'skills', 'certifications', 'languages', 'redrob_signals'])
=== Candidate Quick Profile ===
ID: CAND_0000001
Experience Years: 6.9
Current Location: Toronto
Willing to Relocate: None

=== redrob_signals ===
{'profile_completeness_score': 86.9, 'signup_date': '2025-10-16', 'last_active_date': '2026-05-20', 'open_to_work_flag': True, 'profile_views_received_30d': 23}


# Relocating code

In [9]:
# Converting the whole sample list to a pandas DataFrame
df = pd.DataFrame(sample_data)

print(f"Total initial sample candidates: {len(df)}")

# FILTER 1: Experience Constraint (5 to 9 years)
exp_condition = (df['profile'].apply(lambda x: x.get('years_of_experience', 0)) >= 5) & (df['profile'].apply(lambda x: x.get('years_of_experience', 0)) <= 9)
df_filtered = df[exp_condition]
print(f"Candidates remaining after Experience Filter (5-9 years): {len(df_filtered)}")

# FILTER 2: Location (Noida/Pune or willing to relocate) 
target_cities = ['noida', 'pune']

def matches_location_criteria(row):
    loc = str(row['location']).strip().lower()
    willing_to_relocate = row['willing_to_relocate']
    
    # Condition A: They already live in Noida or Pune
    if loc in target_cities:
        return True
    
    # Condition B: They are willing to relocate
    if willing_to_relocate is True or willing_to_relocate == 1:
        return True
        
    return False

# Apply our location function row by row
def matches_location_criteria(row):
    profile = row.get('profile', {}) or {}
    loc = str(profile.get('location', '')).strip().lower()
    willing_to_relocate = row.get('redrob_signals', {}).get('willing_to_relocate', False)

    if loc in target_cities:
        return True

    if willing_to_relocate is True or willing_to_relocate == 1:
        return True

    return False

location_mask = df_filtered.apply(matches_location_criteria, axis=1)
df_final_filtered = df_filtered[location_mask]

print(f"Candidates remaining after Location Filter: {len(df_final_filtered)}")

Total initial sample candidates: 50
Candidates remaining after Experience Filter (5-9 years): 21
Candidates remaining after Location Filter: 9
